In [ ]:
import numpy as np
import torch
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [8]:
DATA_PATH = '../training_data/dataset.csv'
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
DATA_EMBEDDINGS_PATH = 'data_bert.npz'
DATA_PATH_TEST = '../training_data/teste.csv'

In [10]:
df = pd.read_csv(DATA_PATH)
sentences = df["text"].astype(str).tolist()
labels = df["label"].values

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)


def chunk_text_by_tokens(text, max_tokens=128, overlap=20):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    if len(tokens) <= max_tokens:
        return [text]

    chunks = []
    stride = max_tokens - overlap
    for i in range(0, len(tokens), stride):
        chunk_tokens = tokens[i : i + max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

    return chunks


def extract_bert_embeddings_with_chunks(
    text_list, max_length=128, batch_size=32
):
    bert_model.eval()
    document_embeddings = []

    total_documents = len(text_list)

    with torch.no_grad():
        for doc_idx, text in enumerate(text_list, start=1):

            text_chunks = chunk_text_by_tokens(
                text,
                max_tokens=max_length - 2,
                overlap=20
            )

            chunk_vectors = []

            for i in range(0, len(text_chunks), batch_size):
                batch_chunks = text_chunks[i : i + batch_size]

                inputs = tokenizer(
                    batch_chunks,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt",
                ).to(device)

                outputs = bert_model(**inputs)

                cls_embeddings = (
                    outputs.last_hidden_state[:, 0, :]
                    .cpu()
                    .numpy()
                )

                chunk_vectors.append(cls_embeddings)

            all_chunks_matrix = np.vstack(chunk_vectors)

            # Média dos embeddings dos chunks
            doc_vector = np.mean(all_chunks_matrix, axis=0)
            document_embeddings.append(doc_vector)

            # Progresso
            percent = doc_idx / total_documents * 100

            print(
                f"\rProcessando: {doc_idx}/{total_documents} "
                f"({percent:.1f}%) | "
                f"Chunks: {len(text_chunks)}",
                end="",
                flush=True
            )

    print("\nExtração dos embeddings concluída!")

    return np.array(document_embeddings)

X = extract_bert_embeddings_with_chunks(sentences)
y = np.array(labels)

np.savez_compressed(DATA_EMBEDDINGS_PATH, embeddings=X, labels=y)

print(f"Arquivo {DATA_EMBEDDINGS_PATH} salvo com sucesso!")

Loading weights: 100%|██████████| 199/199 [00:00<?, ?it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processando: 6906/7198 (95.9%) | Chunks: 78

KeyboardInterrupt: 

In [ ]:
if os.path.exists(DATA_EMBEDDINGS_PATH):
    data = np.load(DATA_EMBEDDINGS_PATH)
    X = data["embeddings"]
    y = data["labels"]
else:
    print(f"Arquivo '{DATA_EMBEDDINGS_PATH}' não encontrado. rode a cédula anterior para carregar os dados")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM (RBF)": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

scoring = ['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted']
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

trained_models = {}

**KFOLD**

In [ ]:
for name, clf in classifiers.items():
    scores = cross_validate(clf, X_train, y_train, cv=cv, scoring=scoring)

    print(f"=== {name} (Validação Cruzada - Treino) ===")
    print(f"Acurácia Média: {scores['test_accuracy'].mean():.4f}")
    print(f"F1-Score Médio: {scores['test_f1_weighted'].mean():.4f}")
    print(f"Precisão Média: {scores['test_precision_weighted'].mean():.4f}")
    print(f"Revocação Média: {scores['test_recall_weighted'].mean():.4f}\n")

    clf.fit(X_train, y_train)
    trained_models[name] = clf

=== Logistic Regression (Validação Cruzada - Treino) ===
Acurácia Média: 0.7499
F1-Score Médio: 0.7498
Precisão Média: 0.7505
Revocação Média: 0.7499

=== Random Forest (Validação Cruzada - Treino) ===
Acurácia Média: 0.7516
F1-Score Médio: 0.7515
Precisão Média: 0.7517
Revocação Média: 0.7516



c:\Users\lucas_60v3pre\Documents\Stroj---Resindecia-IA-Puc-Eldorado\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\lucas_60v3pre\Documents\Stroj---Resindecia-IA-Puc-Eldorado\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\lucas_60v3pre\Documents\Stroj---Resindecia-IA-Puc-Eldorado\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


=== SVM (RBF) (Validação Cruzada - Treino) ===
Acurácia Média: 0.7773
F1-Score Médio: 0.7765
Precisão Média: 0.7814
Revocação Média: 0.7773



c:\Users\lucas_60v3pre\Documents\Stroj---Resindecia-IA-Puc-Eldorado\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


=== XGBoost (Validação Cruzada - Treino) ===
Acurácia Média: 0.7571
F1-Score Médio: 0.7567
Precisão Média: 0.7586
Revocação Média: 0.7571

=== LightGBM (Validação Cruzada - Treino) ===
Acurácia Média: 0.7580
F1-Score Médio: 0.7577
Precisão Média: 0.7591
Revocação Média: 0.7580



**TESTE DO MODELO COM O DATASET**

In [ ]:
for name, clf in trained_models.items():
    try:
        y_pred = clf.predict(X_test)

        print(f"=== {name} (Avaliação - Teste) ===")
        #print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
        #print(f"F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
        #print(f"Precisão: {precision_score(y_test, y_pred, average='weighted'):.4f}")
        #print(f"Revocação: {recall_score(y_test, y_pred, average='weighted'):.4f}\n")
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))
    except IndexError as e:
        print(f"!!! Erro ao avaliar o modelo {name}: {e} !!!")

=== Logistic Regression (Avaliação - Teste) ===
Acurácia: 0.7424
F1-Score: 0.7422
Precisão: 0.7432
Revocação: 0.7424

[[389 155]
 [125 418]]
              precision    recall  f1-score   support

           0       0.76      0.72      0.74       544
           1       0.73      0.77      0.75       543

    accuracy                           0.74      1087
   macro avg       0.74      0.74      0.74      1087
weighted avg       0.74      0.74      0.74      1087

=== Random Forest (Avaliação - Teste) ===
Acurácia: 0.7507
F1-Score: 0.7506
Precisão: 0.7509
Revocação: 0.7507

[[400 144]
 [127 416]]
              precision    recall  f1-score   support

           0       0.76      0.74      0.75       544
           1       0.74      0.77      0.75       543

    accuracy                           0.75      1087
   macro avg       0.75      0.75      0.75      1087
weighted avg       0.75      0.75      0.75      1087

=== SVM (RBF) (Avaliação - Teste) ===
Acurácia: 0.7728
F1-Score: 0.771

**TESTE DO MODELO COM O DATASET DE TESTE**

In [ ]:
"""
df_test = pd.read_csv(DATA_PATH_TEST)

X_test = extract_bert_embeddings_with_chunks(df_test["text"].astype(str).tolist())
y_test = df_test["label"].values

for name, clf in trained_models.items():
    try:
        y_pred = clf.predict(X_test)

        print(f"=== {name} (Avaliação - Teste) ===")
        print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
        print(f"F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
        print(f"Precisão: {precision_score(y_test, y_pred, average='weighted'):.4f}")
        print(f"Revocação: {recall_score(y_test, y_pred, average='weighted'):.4f}\n")
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))
    except IndexError as e:
        print(f"!!! Erro ao avaliar o modelo {name}: {e} !!!")
        print(f"O modelo {name} pode não ter sido ajustado corretamente ou não possui estimadores.")
        print("-" * 50)
"""